# 第７章　確率過程の理論計算

宇都宮大学　吉田勝俊

2026.4.27
初版2026.4.21

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm  #3Dプロットのカラーマップ
import sympy as sp  #数式処理
from IPython.display import Code, Math

グラフの線種等の一括設定

In [ ]:
from matplotlib import rcParams
rcParams['lines.linewidth']=0.5
rcParams['lines.markersize']=1.2

テキストセルで使う LaTeX マクロ

- $\newcommand{\ave}[1]{\big\langle#1\big\rangle} \ave{x}$
$\newcommand{\bm}[1]{\boldsymbol{#1}} \bm{x}$
$\newcommand{\red}[1]{\color{red}{#1}} \red{x}$
$\newcommand{\blue}[1]{\color{blue}{#1}} \blue{x}$
$\newcommand{\mat}[1]{\begin{pmatrix}#1\end{pmatrix}} \mat{x_1\\x_2}$
$\newcommand{\bbm}[1]{\bar{\bm #1}} \bbm{x}$

In [ ]:
def display_eq(leftsym, rightsym):
    display(Math( sp.latex(leftsym) + '=' + sp.latex(rightsym) ))

**授業用ライブラリを仮想マシンにインストール**

- [ランタイム](https://docs.cloud.google.com/colab/docs/runtimes?hl=ja)を起動すると，Colab クラウド上に[仮想マシン](https://ja.wikipedia.org/wiki/%E4%BB%AE%E6%83%B3%E3%83%9E%E3%82%B7%E3%83%B3)が作られます．
- ランタイムが終了すると，仮想マシンは削除され，ディスクも消失するので，<br/>次のセルは，ランタイムの起動ごとに実行する必要があります．

In [ ]:
import os
repo_name   =r'kylec'
repo_url    =r'https://github.com/ktysd-lab/kylec.git'
if os.path.isdir(repo_name): #二度読み抑制，アップデートの反映
    %cd {repo_name}
    !git pull
    %cd /content
else:
    !git clone {repo_url}

Cloning into 'kylec'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 49 (delta 22), reused 31 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 9.56 KiB | 2.39 MiB/s, done.
Resolving deltas: 100% (22/22), done.


**確率密度関数 (Probability Density Function) の数式処理用モジュール：**

- `symPDFbase` 基本クラス [[source](https://github.com/ktysd-lab/kylec/blob/main/mcRDS/symPDFbase.py)]
- `symPDFs` 具体的な確率密度関数の実装 [[source](https://github.com/ktysd-lab/kylec/blob/main/mcRDS/symPDFs.py)]

In [ ]:
# from kylec.mcRDS import symPDFbase  #基本クラス(symPDFs と同等なクラスのDIYに必要)
from kylec.mcRDS import symPDFs     #具体的な確率密度関数の実装

**数式処理に使用した Python ライブラリ：**

- [SymPy](https://www.sympy.org/en/index.html)
- [sympyを使って数学をしよう！ #Python - Qiita](https://qiita.com/irisu-inwl/items/45bc2844ecc295e2abe2)

## ■ 問題の定式化

一般に，時間変動するシステムは，微分方程式 $\dot{\bm x}(t) = \bm f\big(\bm x(t)\big)$ か，差分方程式 $\bm x[i] = \bm f\big(\bm x[i-1]\big)$ で数理モデル化されます．<br/>
これらを，**状態空間モデル (SSM, state space model)** と総称し，解を表す $\bm x$ を**状態ベクトル (state vector)** といいます．

本章では，線形の差分方程式を考えます．

### ● 線形差分方程式（ノイズ無し）

線形の差分方程式は，一般に次の形式で書けます．$F$ は行列で，**状態行列 (state matrix)** と呼ばれます．

- $\displaystyle
    \bm x[i] = F\bm x[i-1]
    ,\quad \bm x[0]:=\bm x_0
    ,\quad i=1,\cdots,n.
$

逐次代入によって，次のような解が判明します．

- $
 \bm x[i] = F\bm x[i-1] = F^2\bm x[i-2] = \cdots = F^i\bm x[0] = F^i\bm x_0
$

### ● 線形<u>確率</u>差分方程式（ノイズ有り）

線形差分方程式に，ノイズ $\bm w$ を付加したものを考えます．

- $\displaystyle
    \bm x[i] = F\bm x[i-1] + \bm w[i-1]
    ,\quad \bm x[0]:=\bm x_0
    ,\quad i=1,\cdots,n.
$（**線形確率差分方程式**）

これを，**線形確率差分方程式 (linear stochastic difference equation)** といいます．<br/>
ノイズ無しとの違いとして，
- 状態ベクトル $\bm x[i]$ とノイズ $\bm w[i]$ を，ベクトル確率過程とみなす！



本章では，**線形確率差分方程式のアンサンブル統計を理論計算**し，モンテカルロシミュレーションと比較します．

理論計算の利点として，計算速度が桁違いに速く，所要メモリも桁違いに少なくて済みます．<br/>
このことから，組み込み用途（ドローン，車載コンピュータ）には，理論計算が多用されます（カルマンフィルタ）．

## ■ 準備

そのための予備知識を，いくつか補充します．

### ● 線形写像を介した独立性

> **公式（線形写像を介した独立性）：**
> 確率ベクトル $\bm x$, $\bm y$ が独立ならば，<br/>
> - 線形写像 $A\bm x$ は，$\bm y$ と独立である．
> - 線形写像 $B\bm y$ は，$\bm x$ と独立である．（$A$, $B$ は行列）

**証明：**

線形写像 $\bm y':=B\bm y$ をとると，
$\bm y'$ の関数は，$\bm y$ の関数でもあるので，
- $p(\bm x, \bm y') = p(\bm x, B\bm y) =: p'(\bm x, \bm y)$

となるような確率密度関数 $p'$ を作ることができる．$\bm x$ と $\bm y$ は独立なので，
- $\displaystyle
 p(\bm x|\bm y')
 := \frac{p(\bm x,\bm y')}{p_y(\bm y')}
 = \frac{p'(\bm x,\bm y)}{p_y(\bm y')}
 = \frac{p'_x(\bm x)p'_y(\bm y)}{p_y(\bm y')}
\qquad(1)$

が成立する．ここで，周辺確率密度関数の計算を進めると，
- $\displaystyle
 p_y(\bm y')
 =
 \int_{-\infty}^{\infty}p(\bm x,\bm y') d\bm x
 =
 \int_{-\infty}^{\infty}p'(\bm x,\bm y) d\bm x
 =
 p'_y(\bm y)
\qquad(2)$

- $\begin{aligned}[t]
 p_x(\bm x)
 &= \int_{-\infty}^{\infty}p(\bm x,\bm y') d\bm y'
 = \int_{-\infty}^{\infty}p'(\bm x,\bm y) d\bm y'
 = \int_{-\infty}^{\infty}p'_x(\bm x)p'_y(\bm y) d\bm y'
 \\&
 = p'_x(\bm x)\int_{-\infty}^{\infty}p'_y(\bm y) d\bm y'
 = p'_x(\bm x)\underbrace{\int_{-\infty}^{\infty}\overbrace{p_y(\bm y')}^{式(1)} d\bm y'}_{全確率 = 1}
 = p'_x(\bm x)
 \end{aligned}
$

- ゆえに，$p'_x(\bm x)=p_x(\bm x),\quad p'_y(\bm y)=p_y(\bm y')\qquad(3)$

を得る．式(3) を (1) に代入すると，
- $\displaystyle
 p(\bm x|\bm y')
 = \frac{p'_x(\bm x)p'_y(\bm y)}{p_y(\bm y')}
 = \frac{p_x(\bm x)p_y(\bm y')}{p_y(\bm y')}
 = p_x(\bm x)
$

となり，条件 $\bm y'$ が消滅するので，$\bm y'=B\bm y$ は $\bm x$ と独立である．他の主張も同様に示される．

### ● 共分散行列

今後，ベクトル $\bm x$ や行列 $A$ の平均は，全成分に渡る平均だと定義します．

- $
\bar{\bm x}
=
\ave{\bm x}
=
\Bigg\langle{\mat{x_1\\ \vdots\\ x_n}}\Bigg\rangle
:=
\mat{\ave{x_1}\\ \vdots\\ \ave{x_n}}
$

- $
\bar A
=
\ave{A}
=
\Bigg\langle
    \mat{a_{11} & \cdots & a_{1m}\\ \vdots & \ddots & \vdots\\ a_{n1} & \cdots & a_{nm}\\}
\Bigg\rangle
:=
\mat{\ave{a_{11}} & \cdots & \ave{a_{1m}}\\ \vdots & \ddots & \vdots\\ \ave{a_{n1}} & \cdots & \ave{a_{nm}}\\}
$

#### **》共分散行列の定義：**

２つの確率ベクトル $\bm x=(x_1,\cdots,x_n)^T$ と $\bm y=(y_1,\cdots,y_m)^T$ について，<br/>
各成分間の共分散を，全てリストアップした行列を導入します．

$\displaystyle{}
\Sigma_{xy} :=
\begin{pmatrix}
 \sigma_{xy}[1,1] & \sigma_{xy}[1,2] & \cdots & \sigma_{xy}[1,m] \\
 \sigma_{xy}[2,1] & \sigma_{xy}[2,2] & \cdots & \sigma_{xy}[2,m] \\
 \vdots & \vdots & \ddots & \vdots \\
 \sigma_{xy}[n,1] & \sigma_{xy}[n,2] & \cdots & \sigma_{xy}[n,m] \\
\end{pmatrix}
,\quad
\begin{array}{l}
\bar x_k = \ave{x_k},\quad \bar y_l = \ave{y_l},\\
\sigma_{xy}[k,l] := \ave{(x_k - \bar x_k)(y_l - \bar y_l)}
,\quad
\end{array}
$

この行列を，**共分散行列 (covariance matrix)** といいます．この定義から直ちに，

> **公式（共分散行列の対称性）：**
$\Sigma_{xy} = \Sigma_{xy}^T$　（$^T$ は転置）

#### **》テンソル積による表現：**

定義のままでは，理論計算しにくいので，**テンソル積 (tensor product)** が使われます．

> **定義（テンソル積）:**
>
> - $\displaystyle{}
> \bm x\bm y^T =
> \mat{x_1\\ x_2\\ \vdots\\ x_n}(y_1,y_2,\cdots,y_m)
> \quad:=
> \begin{pmatrix}
>  x_1y_1 & x_1y_2 & \cdots & x_1y_m \\
>  x_2y_1 & x_2y_2 & \cdots & x_2y_m \\
>  \vdots &\vdots &\ddots &\vdots \\
>  x_ny_1 & x_ny_2 & \cdots & x_ny_m \\
> \end{pmatrix}
> $
>
> （$^T$ は転置）

このように，テンソル積とは，２つのベクトルから行列を生成する演算で，<br/>
成分の積の組み合わせを，全てリストアップした行列を生成します．
<br/>テンソル積には，通常の積と同様の分配則と，スカラ倍の結合則が成立します．

- $\bm x(\bm y + \bm z)^T = \bm x\bm y^T + \bm x\bm z^T$,　$(\bm x+ \bm y)\bm z^T = \bm x\bm z^T + \bm y\bm z^T$　（**分配則**）
- $(\alpha \bm x)(\beta\bm y)^T = (\alpha\beta)\bm x\bm y^T$　（**結合則**）

##### **内積 $\bm x^T\bm y$ との関係：**

1. 見た目が紛らわしいですが，転置 $^T$ の場所が違うので，区別できます．
   - **内積：**
$\bm x^T\bm y =
(x_1,x_2,\cdots,x_n)\mat{y_1\\ y_2\\ \vdots\\ y_n}
= x_1y_1 + x_2y_2 + \cdots + x_ny_n
$
   0 ※内積は同次元でないと成立しない．
2. 両者を繋ぐ公式がある（同次元の場合に限る）
   - 内積 $\bm x^T\bm y =\operatorname{trace}(\bm x\bm y^T)$ ※テンソル積の対角成分の総和

> #### 演習１（テンソル積の生成）：
>
> 次のコードは，テンソル積の具体例を生成する．<br/>
> 次元を変えて実行するなど，吟味せよ．
>
> ```python
> def demo_tensor_prod():
>     xvec = sp.symbols("x:4")    #4次元にしとく
>     display_eq('xvec', xvec)
>     yvec = sp.symbols("y:3")    #3次元にしとく
>     display_eq('yvec', yvec)
>
>     print('\n=== 数学の定義通り: xy^T ===')
>     xmat, ymat = sp.Matrix(xvec), sp.Matrix(yvec)   #行列型に変換
>     tensor_prod_math = xmat * sp.transpose(ymat)    #"*" が行列積になる
>     display(tensor_prod_math)
>
>     print('\n=== Sympy のテンソル積 ===')
>     tensor_prod_sympy = sp.tensorproduct(xvec,yvec)
>     display(tensor_prod_sympy)
>
> demo_tensor_prod()
> ```

以上のテンソル積を使うと，共分散行列が，簡潔に表現できます．

$\displaystyle{}
\Sigma_{xy} = \ave{(\bm x - \bar{\bm x})(\bm y - \bar{\bm y})^T}
$

この表現の利点として，テンソル積の分配・結合則が使えるので，次のように展開したりが可能です．

> **公式（共分散行列の展開式）：**
> $\Sigma_{xy}
> = \ave{(\bm x - \bar{\bm x})(\bm y - \bar{\bm y})^T}
> = \ave{\bm x\bm y^T} - \bar{\bm x}\bar{\bm y}^T$

**証明：**<br/>
$\displaystyle{}
\begin{aligned}[b]
\Sigma_{xy}
&=
\ave{(\bm x - \bar{\bm x})(\bm y - \bar{\bm y})^T}
=
\ave{(\bm x - \bar{\bm x})(\bm y^T - \bar{\bm y}^T)}
=
\ave{\bm x\bm y^T - \bm x\bar{\bm y}^T - \bar{\bm x}\bm y^T + \bar{\bm x}\bar{\bm y}^T}
\\&
=
\ave{\bm x\bm y^T} - \ave{\bm x}\bar{\bm y}^T - \bar{\bm x}\ave{\bm y^T} + \bar{\bm x}\bar{\bm y}^T
=
\ave{\bm x\bm y^T} - \bar{\bm x}\bar{\bm y}^T - \bar{\bm x}\bar{\bm y}^T + \bar{\bm x}\bar{\bm y}^T
%\\&
=
\ave{\bm x\bm y^T} - \bar{\bm x}\bar{\bm y}^T
\end{aligned}
$

#### **》自己共分散行列の定義：**

自分自身との共分散行列 $\Sigma_{xx} = \ave{(\bm x - \bar{\bm x})(\bm x - \bar{\bm x})^T}$ のことを，**自己共分散行列 (autocovariance matrix)** といいます．

- $\Sigma_{xx}$ の対角成分は，$\bm x$ の各成分 $x_k$ の分散 $\sigma_x^2[k]=\ave{(x_k-\bar x_k)^2}$ となる．

- 非対角成分は，自己共分散 $\sigma_{xx}[k,l]=\ave{(x_k-\bar x_k)(x_l-\bar x_l)}$ となる．

> #### 演習２（共分散行列の生成）：
>
> 次のコードは，共分散行列と自己共分散行列を生成する．<br/>
> 次元を変えて実行するなど，吟味せよ．
>
> ```python
> def demo_tensor_prod():
>     x = sp.symbols("x:3")   #3次元にしとく
>     mx = sp.symbols("m_x:3")  
>     y = sp.symbols("y:4")   #4次元にしとく
>     my = sp.symbols("m_y:4")  
>
>     xx, yy = sp.Matrix(x), sp.Matrix(y)
>     mxx, myy = sp.Matrix(mx), sp.Matrix(my)
>
>     print('\n=== 共分散行列 <(x-mx)(y-my)> ===')
>     cov = (xx-mxx)*sp.transpose(yy-myy)
>     display(cov)
>
>     print('\n=== 自己共分散行列 <(x-mx)(x-mx)> ===')
>     autocov = (xx-mxx)*sp.transpose(xx-mxx)
>     display(autocov)
>
> demo_tensor_prod()
> ```

### ● 多変数ガウス分布

自己共分散行列 $\Sigma_{xx}$ を使うと，多変数ガウス分布を，次元数によらずに書き下せます．

> **定義（$n$ 変数ガウス分布の確率密度関数）：**
> $n$次元確率ベクトル $\bm x$ について，<br/>
> その平均 $\bar{\bm x}$，自己共分散行列 $\Sigma_{xx}$ を用いて，
> - $\displaystyle{}
> p(\bm x) = N(\bm x, \bar{\bm x}, \Sigma_{xx})
> := \frac{1}{(\pi\sqrt{2})^n\sqrt{|\Sigma_{xx}|}}
> \exp\left(
>   - \frac{(\bm x-\bar{\bm x})^T\Sigma_{xx}^{-1}(\bm x-\bar{\bm x})}{2}
> \right)
> $
>
> （$|\Sigma_{xx}|$ は行列式，$\Sigma_{xx}^{-1}$ は逆行列，$^T$ は転置）

> **定理（ガウス確率ベクトルの変換）：**
> 確率ベクトル $\bm x$,  $\bm y$ がガウス分布ならば，
>
> (1) 可逆な線形変換 $A\bm x$ は，ガウス分布になる．（$B\bm y$ も同じ）<br/>
> (2) スカラ倍 $\alpha\bm x $ は，ガウス分布になる．※(1) の $1$ 次元版<br/>
> (3) 加法 $\bm x + \bm y$ は，ガウス分布になる．

定理 (1) (2) の証明には，次の公式を使います．

> **公式（確率密度関数の変数変換）：**
>
> 確率ベクトル $\bm x$, $\bm y$ に，
> 可逆な線形変換 $\bm y = A\bm x$ が存在するとき，<br/>
> 両者の確率密度関数 $p_x(\bm x)$, $p_y(\bm y)$ は，次の公式で結ばれる．
>
> - $\displaystyle p_y(\bm y)
= \frac{1}{|A|}p_x(A^{-1}\bm y)$

<!-- = p_y(A\bm x) = \frac{1}{|A|}p_x(\bm x)  -->
<!-- - $p_x(\bm x) = p_y(A\bm x)|A|$ -->



**公式の簡易的な証明（2次元）：**　※符号付面積（機械力学テキスト）

$\bm x =(x_1,x_2)$ のある基準点 $\bm x^{*} =(x_1^{*},x_2^{*})$ まわりに，微小な符号付面積 $dx_1\wedge dx_2$ を取る．<br/>
この微小面積に入る確率は，$\bm x$の確率密度関数を使って，次のように書ける．
- $P = p_x(\bm x^{*}) dx_1\wedge dx_2$　（面密度×面積）

対応する $\bm y^{*}=A\bm x^{*}$ まわりに，同様に $dy_1\wedge dy_2$ を取ると，同じ確率は次式でも表せる．
- $
 P
 \begin{aligned}[t]
 &= p_y(\bm y^{*}) dy_1\wedge dy_2
 = p_y(\bm y^{*}) (A_{11}dx_1 + A_{12}dx_2)\wedge(A_{21}dx_1 + A_{22}dx_2)
 \\&
 = p_y(\bm y^{*}) (
    A_{11}A_{22}dx_1\wedge dx_2
    + A_{12}A_{21}dx_2\wedge dx_1)
 \\&
 = p_y(\bm y^{*})(A_{11}A_{22}-A_{12}A_{21})dx_1\wedge dx_2
 = p_y(\bm y^{*})|A|dx_1\wedge dx_2
 \end{aligned}
$

ただし，符号付面積の性質 $dx_i\wedge dx_i=0$, $dx_i\wedge dx_j=-dx_j\wedge dx_i$ を用いた．

$P$ で等置すると，$p_x(\bm x^{*}) dx_1\wedge dx_2 = P = p_y(\bm y^{*})|A|dx_1\wedge dx_2$ より，次式を得る．

- $\displaystyle
p_y(\bm y^{*}) = \frac{1}{|A|} p_x(\bm x^{*})
= \frac{1}{|A|} p_x(A^{-1}\bm y^{*})
$

$n$ 変数の場合は，$|A|$ が $n$ 次行列式となり，同じ公式が成立する．

**定理 (1) (2) の証明：**

(1) ガウス確率ベクトル $\bm x$ の線形変換 $\bm y := A\bm x$ の確率密度関数は，
公式より，

- $\displaystyle{}
p_y(\bm y) = \frac{1}{|A|}p_x(A^{-1}\bm y)
$

となる．変換式の両辺の平均 $\bar{\bm y} := A\bar{\bm x}$ と合せて，公式に代入すると，次式を得る．

- $\displaystyle{}
 p_y(\bm y)
 \begin{aligned}[t]
 &
 = \frac{1}{|A|}\frac{1}{(\pi\sqrt{2})^n\sqrt{|\Sigma_{xx}|}}
 \exp\left(
   - \frac{(\bm y-\bar{\bm y})^T(A^{-1})^T\Sigma_{xx}^{-1}A^{-1}(\bm y-\bar{\bm y})}{2}
 \right)
 \end{aligned}
$

正方行列の行列式の恒等式 $|A|=|A^T|$ を使うと，分母は
$|A|\sqrt{|\Sigma_{xx}|}=\sqrt{|A||\Sigma_{xx}||A^T|}=\sqrt{|A\Sigma_{xx}A^T|}$ と変形できる．これと転置の公式 $(A^{-1})^T=(A^T)^{-1}$ を使うと，次式を得る．
- $\displaystyle{}
 p_y(\bm y)
 \begin{aligned}[t]
 &
 = \frac{1}{(\pi\sqrt{2})^n\sqrt{|A\Sigma_{xx}A^T|}}
 \exp\left(
   - \frac{(\bm y-\bar{\bm y})^T(A\Sigma_{xx}A^T)^{-1}(\bm y-\bar{\bm y})}{2}
 \right)
 = N(\bm y, \bar{\bm y}, A\Sigma_{xx}A^T)
 \end{aligned}
$

すなわち，$\bm y = A\bm x$ は，平均 $\bar{\bm y}=A\bar{\bm x}$，自己共分散行列 $\Sigma_{yy}=A\Sigma_{xx}A^T$ のガウス分布である．

(2) 以上の１次元版なので同様に成立する．

**定理 (3) の強引な証明：** ※エレガントさの欠片もないが，予備知識不要

トリックを使うと，加法は可逆な線形写像で表せるので，成立する．

- トリック：
  - $\bm x$ と $\bm y$ を連結した確率ベクトル $\bm u := (\bm x, \bm y)$ を作る．
  - トリック用の変換行列 $\displaystyle A := \mat{E & E\\ O & X}$ を作る．
    - $E$, $O$ は，$\bm x$, $\bm y$ と同次元の単位行列とゼロ行列，$X$ は $A$ を可逆にする作為的な行列．
- 証明：
  - $\bm x$, $\bm y$ はガウス性（全成分がガウス分布）なので，$\bm u$ もガウス性（全成分がガウス分布）となる．
  - ゆえに，$A\bm u$ の全成分はガウス性になる．
  - $A\bm u$ の上半分は，加法 $\bm x + \bm y$ と同じ成分なので，加法の全成分はガウス性となる．



> #### 演習３（定理 (3) の証明）：
>
> 次のコードは，定理 (3) の証明の具体例を示す．<br/>
> 次元 `dim=3` を変えて実行するなど，吟味せよ．
>
> ```python
> def proof_addition(dim):
>     u = sp.Matrix(
>         sp.symbols(f'x:{dim} y:{dim}')
>     )
>     Eblock = sp.eye(dim)         #単位行列
>     Oblock = sp.zeros(dim,dim)   #ゼロ行列
>     Xblock = sp.diag(*list(range(2,2+dim)))
>     A = sp.Matrix([
>         [Eblock,Eblock],
>         [Oblock,Xblock]
>     ])
>
>     display_eq("u = (x,y)", u)
>     display_eq("A", A)
>     display_eq("det(A)", A.det())
>     display_eq("v = Au", A*u)
>
> proof_addition(dim=3)
> ```
>
> - 実行結果より，行列 $A$ は，行列式 det $A\neq0$より可逆．
> - ゆえに $A\bm u$ の全成分はガウス性となり，その上半分 $\bm x + \bm y$ もガウス性である．

### ● 条件付き確率密度関数による確率過程の分類

確率過程（確率ベクトルの列）について，

- $\bm x[0], \bm x[1], \bm x[2], \cdots, \bm x[n]$

第４章で述べた分類を，条件付き確率密度関数で言い直します．

#### **》一般の確率過程：**

初期状態 $\bm x[0]$ では，確率ベクトルが１つしかないので，その性質は，
- $p(\bm x[0])$　（結合確率密度関数）

で規定されます．次の $\bm x[1]$ は，１つ前の $\bm x[0]$ に依存しうるので，一般には，
- $p(\bm x[1]\,|\,\blue{\bm x[0]})$　（条件付き確率密度関数）

で規定されます．続く $\bm x[2]$ は，$\bm x[1],\bm x[0]$ に依存しうるので，一般には，
- $p(\bm x[2]\,|\,\blue{\bm x[1],\bm x[0]})$　（条件付き確率密度関数）

で規定されます．同様にして，$\bm x[i]$ は，次の条件付き確率密度関数で規定されます．
- $p(\bm x[i]\,|\,\blue{\bm x[i-1],\bm x[i-2],\cdots,\bm x[1],\bm x[0]})$


#### **》無相関過程（第４章）：**

特に，全ての条件が消滅する確率過程を，**無相関過程**といいます．
- $p(\bm x[i]\,|\,\blue{\bm x[i-1],\bm x[i-2],\cdots,\bm x[0]}) = p(\bm x[i])$　（**無相関過程の条件**）

これに連動して，条件付き平均操作からも，条件が消滅します．
- $\displaystyle{}
\ave{\bullet\,|\,\blue{\bm x[i-1],\bm x[i-2],\cdots,\bm x[0]}}
= \ave{\bullet}
$


#### **》マルコフ過程（第４章）：**

直前の条件だけが残る確率過程を，**マルコフ過程**といいます．
- $p(\bm x[i]\,|\,\blue{\bm x[i-1],\bm x[i-2],\cdots,\bm x[0]}) = p(\bm x[i]\,|\,\blue{\bm x[i-1]})$　（**マルコフ過程の条件**）

これに連動して，条件付き平均操作にも，直前の条件だけが残ります．
- $\displaystyle{}
\ave{\bullet\,|\,\blue{\bm x[i-1],\bm x[i-2],\cdots,\bm x[0]}}
= \ave{\bullet\,|\,\blue{\bm x[i-1]}}
$


## ■ 線形確率差分方程式のアンサンブル統計

### ● ガウス入力を受ける線形確率差分方程式

- $\displaystyle
    \bm x[i] = F\bm x[i-1] + \bm w[i-1]
    ,\quad \bm x[0]:=\bm x_0
    ,\quad i=1,\cdots,n.
$　　(★)

ノイズ無しの場合と異なり，状態ベクトル $\bm x[i]$ とノイズ $\bm w[i]$ は確率過程なので，<br/>
確率的な性質を定める必要があります．




#### **》ノイズ $\bm w[i]$ の仮定：**

$\bm w[i]$ は，ガウス白色過程とする．そうなるように，確率分布と相関性を仮定する．

- 平均ベクトル：
   - $\bar{\bm w}[i] = \ave{\bm w[i]} = \bm 0$　（定ベクトル）
- 自己共分散行列：
   - $\Sigma_{ww}[i] = \ave{\bm w[i]\bm w[i]^T} = Q$　（定数行列）
- デルタ相関：
   - $\ave{\bm w[i]\bm w[j]^T} = Q\delta_{ij}$　（第４章）

#### **》状態ベクトル $\bm x[i]$ の仮定：**

- 状態ベクトル $\bm x[i]$ は，ガウス確率ベクトルとし，平均と分散を次のように表記する．
   - 平均ベクトル：　$\bbm x[i] := \ave{\bm x[i]}$

   - 自己共分散行列：　$\Sigma[i] := \ave{(\bm x[i]-\bbm x[i])(\bm x[i]-\bbm x[i])^T}$

- 初期値 $\bm x[i]$ のガウス分布は，既知とする．
   - $\bbm x[0] = \bbm x_0$　（所与の平均ベクトル）

   - $\Sigma[0] = \Sigma_0$　（所与の自己共分散行列）

- 初期値  $\bm x[0]$ は，全ての時刻 $i$ でノイズ $\bm w[i]$ と独立とする．
   - $p\big(\bm x[0],\bm w[i]\big) = p\big(\bm x[0]\big) p\big(\bm w[i]\big)$ となるので，

   - $\ave{\bm x[0]\bm w[i]^T}
    = \ave{\bm x[0]}\underbrace{\ave{\bm w[i]^T}}_{=\bm 0 （ノイズの仮定）} = O
   $ 　（全ての $i\geq 0$ で零行列）


#### **》線形確率差分方程式 (★) のガウス・マルコフ性：**

結論からいうと，線形確率差分方程式 (★) の **解過程** $\bm x[i]$（解を表す確率過程）は，<br/>
**ガウス・マルコフ過程**（ガウス性のマルコフ過程）となります．


- マルコフ性 $\implies$ 現在の状態は，直前の状態だけで決まる．
- ガウス性 $\implies$ 解はガウス分布であり，分布形は平均と分散（自己共分散行列）だけで決まる．

##### **マルコフ性：**

式(★)を見ると，$\bm x[i]$ は，直前の $\bm x[i-1],\bm w[i-1]$ にしか依存していないので，<br/>
解過程 $\bm x[i]$ は，マルコフ過程です．

##### **ガウス性：**

式(★) を時刻 $i=1$ から順に調べてみます．**定理（ガウス確率ベクトルの変換）** を用いると，
- $i=1:$　$\bm x[1] = F\bm x[0] + \bm w[0] = F\bm x_0 + \bm w_0$
  - $F\bm x_0$（ガウス確率ベクトル $\bm x_0$ の線形変換 ）と，$\bm w_0$（ガウス確率ベクトル）の和なので，ガウス性である．
- $i=2:$　$\bm x[2] = F\bm x[1] + \bm w[1] = F^2\bm x_0 + F\bm w_0 + \bm w[1]$
  - やはり，ガウス確率ベクトルの線形変換と加法しかないのでガウス性である．
- $i>3:$　帰納的に，$\bm x[i]$ ガウス性である．

となり，解過程 $\bm x[i]$ は，ガウス性だと分かります．

以上，ガウス性とマルコフ性が両立するので，解過程 $\bm x[i]$ は，**ガウス・マルコフ過程**です．

以上の仮定をまとめると，次の定理が成立します．

> **定理（解過程の性質）：**
>
> (★) の解過程 $\bm x[i]$ は，$\bm w[j]$ ($j\geq i$) と独立な，**ガウス・マルコフ過程**である．
> - $p(\bm x[i]\,|\,\blue{\bm x[i-1],\cdots,\bm x[0],\bm w[i-1],\cdots,\bm w[0]}) = p(\bm x[i]\,|\,\blue{\bm x[i-1]})$

### ● 平均と分散の推移則

解過程 $\bm x[i]$ の分布形は，ガウス過程（ガウス分布）なので，平均と分散（自己共分散行列）だけで決まり，<br/>
マルコフ過程なので，それらは直前の状態だけで決まります．すなわち，

- $\bbm x[i] =$ ( $\bbm x[i-1]$の式 )
- $\Sigma[i] =$ ( $\Sigma[i-1]$の式 )

ゆえに，平均と分散の経時変化は，漸化式で書けるのですが，具体的に求めていきます．

#### **》平均の推移則**

(★): $\bm x[i] = F\bm x[i-1] + \bm w[i-1]$ の両辺の期待値をとる．<br/>
定数行列 $F$ は平均操作に関与しないので，ノイズの仮定 $\ave{\bm w[i]}=\bm 0$ より，次式を得る．

- $\bbm x[i] = \ave{\bm x[i]} = \ave{F\bm x[i-1] + \bm w[i-1]} = F\ave{\bm x[i-1]} + \ave{\bm w[i-1]} = F\ave{\bm x[i-1]} = F\bbm x[i-1]$

すなわち，元の (★) からノイズ項を除いたような形式で，平均の推移則が得られる．

- $\bbm x[i] = F\bbm x[i-1]$　（**平均の推移則**）

#### **》分散（自己共分散行列）の推移則**

解過程の自己共分散行列は，**公式（自己共分散行列の展開式）** より，次のように整理できます．

- $
\begin{aligned}[t]
\Sigma[i] &:= \ave{(\bm x[i] - \bbm x[i])(\bm x[i] - \bbm x[i])^T}
= \ave{\bm x[i]\bm x[i]^T} - \bbm x[i]\bbm x[i]^T
\end{aligned}
$

第 1 項 $\ave{\bm x[i]\bm x[i]^T}$ に (★) を代入し，定数行列 $F$ を外に出すと，次のように展開できます．

- $
\begin{aligned}[t]
\ave{\bm x[i]\bm x[i]^T}
&
= \ave{(F\bm x[i-1] + \bm w[i-1])(F\bm x[i-1] + \bm w[i-1])^T}
\\&
= F\ave{\bm x[i-1]\bm x[i-1]^T}F^T
 + F\ave{\bm x[i-1]\bm w[i-1])^T}
\\&\qquad
 + \ave{\bm w[i-1]\bm x[i-1]^T}F^T
 + \ave{\bm w[i-1]\bm w[i-1])^T}
\end{aligned}
$

 ここで，**定理（解過程の性質）** より，$\bm x[i-1]$ は $\bm w[i-1]$ とノイズの仮定より，

- $
\ave{\bm x[i-1]\bm w[i-1])^T}
= \ave{\bm w[i-1]\bm x[i-1])^T}
= \underbrace{\ave{\bm w[i-1]}}_{ノイズの仮定 =\bm 0}\ave{\bm x[i-1])^T} = O
$ （零行列）
- $\ave{\bm w[i-1]\bm w[i-1])^T}=Q$

これらを，**平均の推移則**とともに代入すると，

- $
\begin{aligned}[t]
\Sigma[i]
&
= \ave{\bm x[i]\bm x[i]^T} - \bbm x[i]\bbm x[i]^T
\\&
= F\ave{\bm x[i-1]\bm x[i-1]^T}F^T + \ave{\bm w[i-1]\bm w[i-1])^T} - F\bbm x[i]\bbm x[i]^TF^T
\\&
= F\ave{(\bm x[i-1]- \bbm x[i])(\bm x[i-1]^T- \bbm x[i])}F^T + Q
\\&
= F\,\Sigma[i-1]F^T + Q
\end{aligned}
$

となり，解過程 $\bm x[i]$ の共分散行列の推移則が得られます．

- $\Sigma[i-1] = F\,\Sigma[i-1]\,F^T + Q$　（**自己共分散行列の推移則**）


以上，理論計算によって，次の公式が得られました．

> **公式（平均と分散の経時変化）：**
>
> (★) の解過程 $\bm x[i]$ について，平均 $\bbm x[i]$ と 自己共分散行列 $\Sigma[i]$ の経時変化は，次式に従う．
>
> - $\bbm x[i] = F\,\bbm x[i-1]$　（**平均の推移則**）
> - $\Sigma[i-1] = F\,\Sigma[i-1]F^T + Q$　（**自己共分散行列の推移則**）


## ■ 数値例

In [ ]:
def plot_stats(stats, style=".-", figax=None, suffix=""):
    """
    アンサンブル統計のプロット
    """
    means, covs = stats
    dim = len(means[0,:])

    if figax is None:
        fig, ax = plt.subplots(2,1,figsize=(6.4,4))
    else:
        fig, ax = figax

    # 平均値のプロット
    for d in range(dim):
        ax[0].plot(means[:,d], style, label=f"$m_{d}${suffix}")

    # 共分散のプロット
    cov_mats = [cov.reshape(dim,dim) for cov in covs]   #行列の復元
    idxs = np.array(np.triu_indices(2,k=0)).T           #上三角の添字
    for i, idx in enumerate(idxs):
        ir, ic = idx
        cov = [cov[ir,ic] for cov in cov_mats]
        ax[1].plot(cov, style, label=r"$\sigma_{%d%d}$%s"%(ir,ic,suffix))

    for a in ax:
        a.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
        a.grid()

    return fig, ax

### ● 実装 ― モンテカルロ・シミュレーション

In [ ]:
def monte_carlo_GaussDE(F, x0, Q, Nsample=21, length=21):
    """
    ガウス線形確率差分方程式 (★) の
    モンテカルロ・シミュレーション
    """

    dim = len(x0)
    # 平均値 mean と自己共分散行列 cov の時系列
    means = np.zeros((length, dim))
    covs  = np.zeros((length, dim*dim)) #重複する非対角要素が無駄だが

    # xi の標本（初期値）
    xi_sample = np.array([x0 for j in range(Nsample)])

    for i in range(length):

        ### 平均と自己共分散行列
        mean_vec = np.mean(xi_sample, axis=0) #行方向が標本番号
        cov_mat  = np.cov(xi_sample.T) #行方向が標本番号
        #配列にストア
        means[i,:] = mean_vec
        covs[i,:]  = cov_mat.reshape(dim*dim)
        if not i<length:
            break

        ### 時刻 i の標本の取得
        for j in range(1,Nsample):
            # ノイズ（ガウス白色過程，平均0, 自己共分散行列Q）
            w = np.random.multivariate_normal(mean=np.zeros(dim), cov=Q)
            # 線形確率差分方程式
            xold = xi_sample[j,:]
            xi = np.dot(F, xold) + w
            # 標本の更新
            xi_sample[j,:] = xi

    return means, covs

> #### 演習４（★のモンテカルロ・シミュレーション）：
>
> 次のコードは，ガウス線形確率差分方程式 (★) の解過程について，<br/>
> モンテカルロ・シミュレーションにより，平均，分散，共分散を近似計算する．
> 1. 以下のセルで何度か実行し，結果のばらつきを観察せよ．
> 2. 標本数 `Nsample=` を増加させて，結果の再現性を高めよ．
> 3. 標本数の増加によって，計算時間がどの程度，悪化するか述べよ．
>
> ```python
> F = np.array([  #状態行列
>     [0, 0.5],
>     [-0.8, 0]
> ])
> x0 = np.array(  #状態の初期値
>     [0.1, 0]
> )
> Q = np.array([  #ノイズの自己共分散行列
>     [0.03, 0.02],
>     [0.02, 0.03]
> ])
> Nsample = 201   #標本路数
> length  = 21    #標本路長
>
> %time stats_monte = monte_carlo_GaussDE(F, x0, Q, Nsample, length)
>
> plot_stats(stats_monte);
>```

### ● 実装 ― 公式（平均と自己共分散行列の経時変化）による理論計算

In [ ]:
def theoretical_GaussDE(F, x0, Q, Nsample=21, length=21):
    """
    ガウス線形確率差分方程式 (★) の
    理論計算
    """

    dim = len(x0)
    # 平均値 mean と自己共分散行列 cov の時系列
    means = np.zeros((length, dim))
    covs  = np.zeros((length, dim*dim)) #重複する非対角要素が無駄だが

    # 初期値
    mean_vec = x0.copy()
    cov_mat  = np.zeros((dim,dim))

    for i in range(length):

        #配列にストア
        means[i,:] = mean_vec
        covs[i,:]  = cov_mat.reshape(dim*dim)
        if not i<length:
            break

        ### 公式
        mean_old    = mean_vec
        cov_old     = cov_mat

        mean_vec = F.dot(mean_old)
        cov_mat  = F.dot(cov_old).dot(F.T) + Q

    return means, covs

> #### 演習５（★の理論計算）：
>
> 次のコードは，ガウス線形確率差分方程式 (★) の解過程について，<br/>
> **公式 (平均と分散の経時変化)** により，平均，分散，共分散を，厳密に計算する．
> 1. 以下のセルで何度か実行し，結果のばらつきを観察せよ．
> 2. モンテカルロシミュレーションと比べて，計算時間がどのぐらい短縮されたか考察せよ．
> 3. 理論計算に匹敵する精度を，モンテカルロシミュレーションで実現するには，どの程度の標本数が必要か，検証せよ．
>
> ```python
> F = np.array([  #状態行列
>     [0, 0.5],
>     [-0.8, 0]
> ])
> x0 = np.array(  #状態の初期値
>     [0.1, 0]
> )
> Q = np.array([  #ノイズの自己共分散行列
>     [0.03, 0.02],
>     [0.02, 0.03]
> ])
>
> %time stats_theory = theoretical_GaussDE(F, x0, Q, Nsample, length)
>
> fig, ax = plot_stats(stats_theory, style="-")
> plot_stats(stats_monte, style="x", figax=(fig,ax), suffix=" (monte)")
> for a in ax: a.grid()
>```